# value_and_grad with an auxiliary metric

`jax.grad` turns a scalar function into its gradient function. In training you want the loss value too, and often a metric computed from the same forward pass. `jax.value_and_grad(f, has_aux=True)` returns all of them from one pass.

Write `loss_acc_grads(params, x, y, l2)` for a softmax classifier:

- `params`: dict `{'w': (D, K), 'b': (K,)}`.
- `x`: `(N, D)` float; `y`: `(N,)` int labels in `[0, K)`; `l2`: float.
- Logits $z = x W + b$. Loss:

$$\mathcal{L} = \frac{1}{N}\sum_{n} -\log \mathrm{softmax}(z_n)_{y_n} \;+\; \frac{l_2}{2}\lVert W \rVert_F^2$$

  The penalty applies to `w` only, not `b`.
- Accuracy: the fraction of rows whose `argmax` logit equals the label (the lowest index wins a tie).

Return `(loss, acc, grads)`: two float32 scalars and a dict `{'w': (D, K), 'b': (K,)}` holding $\partial \mathcal{L}/\partial W$ and $\partial \mathcal{L}/\partial b$. The accuracy carries no gradient. Use one call to `jax.value_and_grad`.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def loss_acc_grads(params, x, y, l2):
    """params {'w': (D, K), 'b': (K,)}. Return (loss, acc, grads dict)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__dict__": [["w", {"__nd__": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float64"}], ["b", {"__nd__": [0.0, 0.0, 0.0], "dtype": "float64"}]]}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [0, 2], "dtype": "int64"}, 0.0], "e": {"__tup__": [{"__nd__": 1.0986123085021973, "dtype": "float32"}, {"__nd__": 0.5, "dtype": "float32"}, {"__dict__": [["b", {"__nd__": [-0.166666641831398, 0.3333333432674408, -0.166666641831398], "dtype": "float32"}], ["w", {"__nd__": [[-0.3333333134651184, 0.1666666716337204, 0.1666666716337204], [0.1666666716337204, 0.1666666716337204, -0.3333333134651184]], "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["w", {"__nd__": [[1.0, -1.0], [0.5, 0.5]], "dtype": "float64"}], ["b", {"__nd__": [0.0, 0.1], "dtype": "float64"}]]}, {"__nd__": [[1.0, 2.0]], "dtype": "float64"}, {"__nd__": [1], "dtype": "int64"}, 0.1], "e": {"__tup__": [{"__nd__": 2.164386749267578, "dtype": "float32"}, {"__nd__": 0.0, "dtype": "float32"}, {"__dict__": [["b", {"__nd__": [0.8698915839195251, -0.8698915243148804], "dtype": "float32"}], ["w", {"__nd__": [[0.9698916077613831, -0.9698915481567383], [1.7897831201553345, -1.6897830963134766]], "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["w", {"__nd__": [[-0.6517911526116896, -0.17471729232577715, 1.6637239913911968], [0.659147749832255, -1.6413972945846467, -0.005203264171931977], [-0.6234637409883934, 0.14863152325202633, -1.608187784186389], [0.2417718768768513, 0.23538091873745476, 1.5756260314314627]], "dtype": "float64"}], ["b", {"__nd__": [0.3166450164719021, 0.5105466616976417, -1.4931166849642326], "dtype": "float64"}]]}, {"__nd__": [[2.2527291247240275, -1.9156455579583005, 1.1018018558224842, -0.3299040738738497], [-0.8806465179277969, -0.6562850109290311, -0.6720146806586559, 0.3801890922489917], [-0.11005869527805158, 1.482571163033857, -1.8296040234453508, -0.0030809646658509597], [-0.892055788321591, 0.7758718474580595, -2.118060042185068, -0.3436943500994811], [0.21017292733610196, -1.4842685839963512, 0.9852155828581877, 0.1787134291183962], [1.006856770707084, 0.9591848586745111, -0.9798545663746768, -0.7977957578895382], [-0.20333248577190258, 0.7479597259631346, 0.8509672909755427, -0.7096848707086849], [-0.6071971852553313, -0.7978229151665286, -0.584238229157347, -0.2379233775647261]], "dtype": "float64"}, {"__nd__": [0, 1, 1, 2, 0, 0, 0, 2], "dtype": "int64"}, 0.0], "e": {"__tup__": [{"__nd__": 3.026244878768921, "dtype": "float32"}, {"__nd__": 0.375, "dtype": "float32"}, {"__dict__": [["b", {"__nd__": [-0.06691427528858185, 0.2172786295413971, -0.15036435425281525], "dtype": "float32"}], ["w", {"__nd__": [[-0.5228826999664307, 0.2875667214393616, 0.23531606793403625], [0.5248939394950867, -0.6123660802841187, 0.08747205138206482], [-0.6919691562652588, 0.47004806995391846, 0.2219211459159851], [0.06015053391456604, -0.08718399703502655, 0.027033474296331406]], "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["w", {"__nd__": [[0.4585777290940279, -0.9530740655699254, -0.36862714912240996], [0.013318481161013024, 0.7741459470172646, -1.3160148587467566], [1.3714694572870232, -0.35245736590160387, 0.1694164211036251], [0.8470883039345106, 0.6607929217270263, 1.059231886175497]], "dtype": "float64"}], ["b", {"__nd__": [0.17319781643187807, -0.019613353374506273, 0.3164785392796004], "dtype": "float64"}]]}, {"__nd__": [[-0.9957153901865851, 1.2141873689841987, -0.7753862486410281, -1.2602438798825923], [2.0564087809385283, -0.13644493951285103, -1.179124872329948, 1.8522697201129288], [-0.3297126244250999, 1.0620757877855616, -0.8294887233474758, -0.24878002125463117], [-1.6885293534763577, -1.908939358835272, -0.9764345880627332, -0.07803901853109632], [0.7979857101860429, 2.426220501220823, -1.0318629950946103, 1.0090402997688306], [-0.6043409584858992, -0.15252930610285104, -1.4811916182972398, -1.4789857368704227], [-0.7784606028005082, -1.1931106223006203, -0.0035543906764923003, -0.7025144637985207], [1.2506970737478889, 0.153019711281495, 0.1652381892572177, 1.137435198692834]], "dtype": "float64"}, {"__nd__": [0, 0, 1, 2, 0, 1, 1, 1], "dtype": "int64"}, 0.5], "e": {"__tup__": [{"__nd__": 3.5428061485290527, "dtype": "float32"}, {"__nd__": 0.375, "dtype": "float32"}, {"__dict__": [["b", {"__nd__": [-0.20402246713638306, -0.033421918749809265, 0.2374444603919983], "dtype": "float32"}], ["w", {"__nd__": [[0.19157305359840393, -0.5307164192199707, -0.0924183577299118], [-0.39427822828292847, 0.888655424118042, -0.75865238904953], [0.9917906522750854, -0.3554251194000244, -0.04215121269226074], [0.40083742141723633, 0.31094491481781006, 0.5717742443084717]], "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["w", {"__nd__": [[10.76721956055248, 7.830856166867024, 3.6591795790617554, 7.988293270822742, -5.445332988553511], [5.110228046676667, -14.123336629703386, 13.472705522559004, 0.9004825886490181, 1.8339300077143716], [5.99383845795459, -0.09709574361003655, -2.5410788491596263, 11.897846220814097, 1.1575067070989449]], "dtype": "float64"}], ["b", {"__nd__": [0.0, 0.0, 0.0, 0.0, 0.0], "dtype": "float64"}]]}, {"__nd__": [[-0.9158716729607007, 0.4395589157792016, -1.7051204421874975], [1.6555424372356622, -0.1711311122004771, 0.04588385344649914], [2.2577704585568408, 2.1517410791156424, 0.11580667996735006], [0.6248663505788951, 1.09017986530538, 1.6820210064715346], [-0.13109289942963767, 0.13526021392558873, 1.2246342222815874], [0.26955030257702084, 0.7544998572882148, 0.6098595259521409]], "dtype": "float64"}, {"__nd__": [4, 1, 4, 4, 3, 1], "dtype": "int64"}, 0.01], "e": {"__tup__": [{"__nd__": 20.387413024902344, "dtype": "float32"}, {"__nd__": 0.1666666716337204, "dtype": "float32"}, {"__dict__": [["b", {"__nd__": [0.2684297561645508, -0.3111918568611145, 0.31395775079727173, 0.22150182723999023, -0.492697536945343], "dtype": "float32"}], ["w", {"__nd__": [[0.4685041308403015, -0.20588409900665283, 0.17173080146312714, 0.20258019864559174, -0.38892894983291626], [0.18938083946704865, -0.24225059151649475, 0.48967936635017395, 0.22716060280799866, -0.5920301675796509], [0.1262584924697876, -0.10924557596445084, -0.2623768150806427, 0.4257856607437134, -0.01631159335374832]], "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["w", {"__nd__": [[-0.2601207042167583, 1.338837428330753], [-1.1092067436790745, 1.2338035734185084], [0.9995461729736715, -0.05484218663972172], [-0.007611643764543189, 0.08283314085951675], [-0.8610869137790327, -0.28233040463147097], [-0.04656430657923056, 0.611937945639494]], "dtype": "float64"}], ["b", {"__nd__": [3.0, -3.0], "dtype": "float64"}]]}, {"__nd__": [[0.7918128456176918, -0.7427380514871819, 0.825713141083434, -0.7778705404988745, -0.16891653297744932, 0.36950535567059684], [-0.02265339704007352, -0.12696083044176004, 0.2694412578838159, -0.5527990484546487, -0.6526661441559254, -0.20411410779129852], [0.9612712123960654, -0.1016680803594762, 1.904521332881238, 0.2142532717506217, -2.974398618140543, -0.43955369675751277], [1.6650341073744352, 0.006345834984433874, -0.43783367636909615, -0.8736597389981599, 1.3237978282395066, 1.0028245269859126], [0.6720348492501695, 0.4281019403409819, 1.4409062530134493, 0.8212017023979049, -1.2298004032252794, 0.7684435399175684], [0.1696263351170497, -0.11031979370766849, 0.7633881282555145, 0.028386505149749598, -2.933554853997736, 1.5573881199880404], [1.03772000140565, -0.12258922086782367, -0.3723966290308431, 0.2266484269016432, -1.2843938267586974, 0.3064056636400867], [-0.2301970607786508, -0.352950842362128, -0.9231185558488285, -0.6786362006544445, 1.106296434958061, 1.3350210328283845], [0.5028380708910459, -0.04553661850622734, -0.8779033831785013, 0.7177536791417223, 1.6109361945949745, -1.6145039542817976], [1.1742593021682028, -2.165669653480299, -0.3400090763484035, 0.3846246121344049, -1.3649870946709435, -1.5169907262007192], [0.18567376946473443, 0.8767150309491412, -0.4754530494395559, 1.3438920127491554, -0.7799220663001268, 1.7543514042691053], [0.12040500880461658, 0.2353033058064019, 0.6149357025025846, 0.3210268566874286, 1.8457803050035104, 2.206483063408338], [0.5461177994130937, -0.16570477478287973, -1.1761006990541674, -0.7543747264549445, 0.009248189011741795, -0.10133885706584574], [-0.04367359861427566, 0.005048451029825634, 0.49553972212029984, -1.3506872052058774, -1.0254756280086132, 1.1219208705156984], [0.8382925391554815, -0.5357815089844378, 1.295333980594746, -0.6763912647197142, -0.25179208412346116, -1.8254710649362071], [0.8471222673650767, -0.008751091283308206, 1.0697831154945725, 0.7092646584243003, 0.09947106882979134, 0.16600695337323335], [1.202069671308528, -0.15481203854891207, -0.14227878584071332, 0.08018752414269664, 0.818736303952953, 0.9381846358677198], [1.934497217527141, -0.05894193951479713, 1.2884654366703447, -0.6258324239768248, 0.6350174895697084, 0.15966722635574604], [1.7719189395189767, 0.6483385275977063, -0.40744478373891585, 0.6204893353394123, 0.3898468357353195, -0.27225052162371727], [-0.24905620767605252, -0.8976904193941799, 2.8043319412036705, 0.18290134186419704, 1.0226532132010455, -1.1169905865894318]], "dtype": "float64"}, {"__nd__": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "dtype": "int64"}, 0.0], "e": {"__tup__": [{"__nd__": 0.03606295958161354, "dtype": "float32"}, {"__nd__": 1.0, "dtype": "float32"}, {"__dict__": [["b", {"__nd__": [-0.033003345131874084, 0.03300332650542259], "dtype": "float32"}], ["w", {"__nd__": [[-0.04335273802280426, 0.04335273057222366], [-0.011720008216798306, 0.011720017530024052], [0.010338898748159409, -0.010338902473449707], [-0.005823659710586071, 0.005823660176247358], [-0.01901201717555523, 0.019012005999684334], [-0.019220497459173203, 0.019220497459173203]], "dtype": "float32"}]]}]}}, {"a": [{"__dict__": [["w", {"__nd__": [[-1.0368651406568785, -0.10387420939642665, -0.09927146856704905, 0.8522030185301346], [-0.9537264726805303, 0.5900864341051688, 1.5655534503057242, -0.8636458732292807], [0.9604621253114379, -0.9099019851565014, 0.39696248648137683, 0.41418584396020036], [1.7882879160024885, -0.6398277806849743, 0.21922658016656438, 1.0945159177456507], [-1.1212269270861612, 0.04168013510434051, -0.8194099656011558, -0.6514171909737703]], "dtype": "float64"}], ["b", {"__nd__": [0.7324393207508331, 0.7509714114673515, 2.628106119344124, 1.4274554265620245], "dtype": "float64"}]]}, {"__nd__": [[1.3393956530093372, -0.11676380200040161, -0.25091788597956105, 0.18602922978100997, -2.2267565397809213], [1.7047677751212658, -0.6246207268211226, -1.0151551380348043, -0.414721644524359, 0.9168834151535245], [1.2834435014980463, -0.3383313852927848, 0.47303017539746206, -0.021620125439037064, -0.962156349736473], [0.2026726183568746, 0.05913431606857885, -1.5071538681636054, 1.1297919219231545, -1.54315828186273], [0.2612730471866325, -1.023514407472337, -0.7883965094444757, -0.6774150886909966, 0.7292334231560876], [-1.2306690905524644, -0.16769966254406435, 1.5742115977799505, -0.6745728165879054, -0.011189795575854427], [0.68418112970563, -0.5636993991014299, 0.49371721742131663, -0.6430266177294371, 0.7172439574798849], [0.07942344483599749, -0.9195491696517337, 1.4256374334746589, -0.07857938585982074, -0.2637220050702941], [-0.9088499680269638, -0.4542069815799693, -0.08077348387943, 0.1873017343712804, 1.1133844988399955], [-1.3032360616193475, -0.33489386223019485, 1.2104649069395903, 0.19580209601547094, 0.42199900504313625], [1.6066585628255345, 0.3509720553612781, 0.47962821970188263, -0.027718929155257546, 1.7806575945572853], [1.0627810970389353, -0.7088897926970587, 0.5604350668562151, -1.1366224937006022, 1.0983201942238328], [0.6820676743368, -0.15016854176968703, -0.3929110439950159, -1.7868228381220084, 0.7709866409742615], [-0.11046920726424042, -0.3056748375395381, 0.7284855506966182, -0.660247103532902, -0.5830860007199183], [-0.6914806092799765, -1.3511924911654463, 2.4107328154241845, -0.19409785981244757, -0.350515950236606], [0.8996481287639838, -3.150962810068764, -0.03327301850577713, 0.00921326453511899, -0.9763487683390772], [0.45315886512250836, -2.0350808735107173, -0.009058926504619405, 0.860968824740586, -2.254921892315956], [-1.491550021523633, -0.2125523666267897, -0.08649378209822955, -0.7290129838619026, 0.26733953880526645], [0.13792045998173602, -1.4635001101370253, -0.22432641960401753, -1.4088387394353237, -2.2176812473298546], [-2.560515372904948, 0.8732164923565644, -0.2883062753863353, 1.7932034407381086, -1.6309246504259265], [-0.5253295102654808, 0.16670386828735514, -0.3159458650548306, 0.32897590048263653, -0.024797827106022765], [-0.7837550496941335, -2.497628287360758, -1.1147848479180842, 1.5594944669495905, 0.6082985690963003], [1.2233882523452837, -0.7364241490707799, -0.338126487708947, -1.0197584663966615, 0.25239397991881257], [-0.07351211225974372, 0.3204063237704337, 1.6324464310546851, -0.9643092974601902, 0.6115328274260176], [-1.528056605878365, 0.9799928626949387, -0.5650072775362897, -0.19473381102391252, 0.014454002985211167], [1.5250083748441063, -0.15312271871256172, -0.6194479836144084, -1.3061418365796362, 0.15439478723373204], [-0.9767105115916423, -0.625129306958018, -0.1472251885572021, 0.5501624607963658, -0.8369993509375739], [0.5488540249232975, 1.6492625200113638, -1.6116336210579334, 1.2413233827202526, 2.0329701740793213], [-0.7120074138687121, 0.4937293390675241, -1.4944211915357009, 0.22011544597662686, -0.44565012132616116], [0.18508723379763697, -1.0388377193842937, -0.944583123346283, -0.5717085705302731, -0.8385680455394496], [-0.7484551553556026, -0.3347692178533813, -1.2459325074518615, -0.2837437926521366, 0.34184247290082903], [-2.4557788779715435, 0.053760878211953654, -0.8923131647885544, -1.269245755517086, -1.4974932265087706]], "dtype": "float64"}, {"__nd__": [1, 0, 2, 1, 0, 3, 1, 2, 2, 2, 2, 2, 0, 2, 0, 1, 3, 1, 3, 0, 3, 3, 0, 0, 2, 3, 1, 2, 1, 3, 1, 2], "dtype": "int64"}, 0.001], "e": {"__tup__": [{"__nd__": 2.138183355331421, "dtype": "float32"}, {"__nd__": 0.3125, "dtype": "float32"}, {"__dict__": [["b", {"__nd__": [0.004228636156767607, -0.08116266131401062, 0.035468216985464096, 0.041465822607278824], "dtype": "float32"}], ["w", {"__nd__": [[-0.1915082484483719, 0.05593907833099365, -0.0368165485560894, 0.17199790477752686], [-0.09923624992370605, 0.10122325271368027, 0.019838228821754456, -0.021486973389983177], [0.03352939710021019, 0.03264714032411575, -0.0976182296872139, 0.03230336681008339], [0.18975311517715454, -0.12129738926887512, -0.03694115951657295, -0.029052376747131348], [-0.16067618131637573, 0.2116796225309372, -0.10933986306190491, 0.05578606575727463]], "dtype": "float32"}]]}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(loss_acc_grads, json.loads(_TESTS), 'jax', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")